# 12 — ViTPose-Huge 384x288 (Colab)

Sinh tự động bởi `make_step7_notebook.py` — đừng sửa code ở các ô %%writefile.
Cách dùng: sửa **ô 2 (CONFIG)** nếu cần → **Run All** → chạy xong **Ctrl+S**.
Mất kết nối giữa chừng: chọn lại kernel Colab và Run All — train tự chạy tiếp từ checkpoint trên Drive.

In [ ]:
# 1. Setup: Drive, dữ liệu, GPU
import json, os, shutil, subprocess, sys, time, zipfile
import torch

DRIVE_DIR = "/content/drive/MyDrive/upar_track3"
WORK, DATA = "/content/upar", "/content/data"
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
if not os.path.isfile(f"{DATA}/annotations/train_upar_all_fixed.json"):
    shutil.copy(f"{DRIVE_DIR}/upar_track3_train.zip", "/content/upar_track3_train.zip")
    with zipfile.ZipFile("/content/upar_track3_train.zip") as zf:
        zf.extractall(DATA)
import cv2, safetensors, huggingface_hub  # noqa: F401  (all preinstalled on Colab)
p = torch.cuda.get_device_properties(0)
print(f"GPU {p.name} {p.total_memory / 1e9:.0f} GB | CPUs {os.cpu_count()} | torch {torch.__version__} | cv2 {cv2.__version__}")

In [ ]:
%%writefile /content/upar/vitpose_model.py
"""ViTPose (plain ViT backbone + classic deconvolution head) in pure PyTorch.

Used both for training on Colab and inside the submission (torch 2.4.1, no
transformers/mmpose), so it only relies on torch. Weights can be loaded from
the Hugging Face port (usyd-community/vitpose-base, COCO only) or from the
MMPose 1.x COCO checkpoints (td-hm_ViTPose-{base,large,huge}_..._coco-256x192).
ViTPose+ (mixture-of-experts, multi-dataset) checkpoints are rejected: the
Track 3 rules only allow COCO / PoseTrack18 pose pre-training.
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

PRESETS = {
    "base": dict(embed_dim=768, depth=12, num_heads=12),
    "large": dict(embed_dim=1024, depth=24, num_heads=16),
    "huge": dict(embed_dim=1280, depth=32, num_heads=16),
}

# PoseTrack18 slot -> COCO channel used to initialise the head (slots 1/2 have no
# COCO counterpart, start them from the nose channel).
PT18_FROM_COCO = (0, 0, 0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16)


class DropPath(nn.Module):
    def __init__(self, p: float = 0.0) -> None:
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


class Attention(nn.Module):
    def __init__(self, dim: int, num_heads: int) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, c = x.shape
        qkv = self.qkv(x).reshape(b, n, 3, self.num_heads, c // self.num_heads).permute(2, 0, 3, 1, 4)
        x = F.scaled_dot_product_attention(qkv[0], qkv[1], qkv[2])
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class Block(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float, drop_path: float, eps: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=eps)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=eps)
        self.fc1 = nn.Linear(dim, int(dim * mlp_ratio))
        self.fc2 = nn.Linear(int(dim * mlp_ratio), dim)
        self.drop_path = DropPath(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop_path(self.attn(self.norm1(x)))
        return x + self.drop_path(self.fc2(F.gelu(self.fc1(self.norm2(x)))))


class ViTPose(nn.Module):
    def __init__(self, embed_dim: int = 768, depth: int = 12, num_heads: int = 12, mlp_ratio: float = 4.0,
                 img_size: tuple[int, int] = (256, 192), patch_size: int = 16, num_keypoints: int = 17,
                 drop_path_rate: float = 0.0, deconv_channels: int = 256, eps: float = 1e-6) -> None:
        super().__init__()
        self.depth = depth
        self.grad_checkpoint = False  # recompute block activations in backward (saves memory at 384x288)
        self.patch_embed = nn.Conv2d(3, embed_dim, patch_size, stride=patch_size, padding=2)
        hp = (img_size[0] + 4 - patch_size) // patch_size + 1
        wp = (img_size[1] + 4 - patch_size) // patch_size + 1
        self.grid = (hp, wp)
        self.pos_embed = nn.Parameter(torch.zeros(1, hp * wp + 1, embed_dim))
        rates = [drop_path_rate * i / max(depth - 1, 1) for i in range(depth)]
        self.blocks = nn.ModuleList(Block(embed_dim, num_heads, mlp_ratio, r, eps) for r in rates)
        self.last_norm = nn.LayerNorm(embed_dim, eps=eps)
        self.deconv1 = nn.ConvTranspose2d(embed_dim, deconv_channels, 4, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(deconv_channels)
        self.deconv2 = nn.ConvTranspose2d(deconv_channels, deconv_channels, 4, stride=2, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(deconv_channels)
        self.final = nn.Conv2d(deconv_channels, num_keypoints, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        b, c, hp, wp = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed[:, 1:] + self.pos_embed[:, :1]
        for block in self.blocks:
            if self.grad_checkpoint and self.training:
                x = torch.utils.checkpoint.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        x = self.last_norm(x).transpose(1, 2).reshape(b, c, hp, wp)
        x = F.relu(self.bn1(self.deconv1(x)))
        x = F.relu(self.bn2(self.deconv2(x)))
        return self.final(x)

    def layer_id(self, name: str) -> int:
        """Depth index for layer-wise learning-rate decay."""
        if name.startswith(("patch_embed", "pos_embed")):
            return 0
        if name.startswith("blocks."):
            return int(name.split(".")[1]) + 1
        return self.depth + 1


def build(preset: str, **kwargs) -> ViTPose:
    return ViTPose(**PRESETS[preset], **kwargs)


def build_for_config(cfg: dict, **kwargs) -> ViTPose:
    """Model matching an exported model_config.json (preset + input size; old configs are 256x192)."""
    img_size = (int(cfg.get("input_height", 256)), int(cfg.get("input_width", 192)))
    return build(cfg["preset"], img_size=img_size, **kwargs)


# ---------------------------------------------------------------- weight conversion

def _head_from(src: dict, prefix: dict) -> dict:
    return {
        "deconv1.weight": src[prefix["deconv1"] + ".weight"],
        "deconv2.weight": src[prefix["deconv2"] + ".weight"],
        **{f"bn1.{k}": src[f"{prefix['bn1']}.{k}"] for k in ("weight", "bias", "running_mean", "running_var")},
        **{f"bn2.{k}": src[f"{prefix['bn2']}.{k}"] for k in ("weight", "bias", "running_mean", "running_var")},
        "final.weight": src[prefix["final"] + ".weight"],
        "final.bias": src[prefix["final"] + ".bias"],
    }


def from_hf(src: dict, depth: int) -> dict:
    """Hugging Face VitPoseForPoseEstimation (non-MoE) state dict -> ViTPose."""
    if any("experts" in k for k in src):
        raise ValueError("ViTPose+ (MoE) checkpoint: not allowed by the Track 3 rules")
    out = {
        "patch_embed.weight": src["backbone.embeddings.patch_embeddings.projection.weight"],
        "patch_embed.bias": src["backbone.embeddings.patch_embeddings.projection.bias"],
        "pos_embed": src["backbone.embeddings.position_embeddings"],
        "last_norm.weight": src["backbone.layernorm.weight"],
        "last_norm.bias": src["backbone.layernorm.bias"],
    }
    for i in range(depth):
        p = f"backbone.encoder.layer.{i}."
        a = p + "attention.attention."
        out[f"blocks.{i}.attn.qkv.weight"] = torch.cat([src[a + "query.weight"], src[a + "key.weight"], src[a + "value.weight"]])
        out[f"blocks.{i}.attn.qkv.bias"] = torch.cat([src[a + "query.bias"], src[a + "key.bias"], src[a + "value.bias"]])
        out[f"blocks.{i}.attn.proj.weight"] = src[p + "attention.output.dense.weight"]
        out[f"blocks.{i}.attn.proj.bias"] = src[p + "attention.output.dense.bias"]
        for mine, theirs in (("norm1", "layernorm_before"), ("norm2", "layernorm_after"), ("fc1", "mlp.fc1"), ("fc2", "mlp.fc2")):
            out[f"blocks.{i}.{mine}.weight"] = src[p + theirs + ".weight"]
            out[f"blocks.{i}.{mine}.bias"] = src[p + theirs + ".bias"]
    out.update(_head_from(src, {"deconv1": "head.deconv1", "deconv2": "head.deconv2", "bn1": "head.batchnorm1",
                                "bn2": "head.batchnorm2", "final": "head.conv"}))
    return out


def from_mmpose(src: dict, depth: int) -> dict:
    """MMPose 1.x ViTPose COCO checkpoint (mmpretrain VisionTransformer backbone + HeatmapHead) -> ViTPose."""
    src = src.get("state_dict", src)
    if any("experts" in k for k in src):
        raise ValueError("ViTPose+ (MoE) checkpoint: not allowed by the Track 3 rules")
    out = {
        "patch_embed.weight": src["backbone.patch_embed.projection.weight"],
        "patch_embed.bias": src["backbone.patch_embed.projection.bias"],
        "pos_embed": src["backbone.pos_embed"],
        "last_norm.weight": src["backbone.ln1.weight"] if "backbone.ln1.weight" in src else src["backbone.pre_norm.weight"],
        "last_norm.bias": src["backbone.ln1.bias"] if "backbone.ln1.bias" in src else src["backbone.pre_norm.bias"],
    }
    for i in range(depth):
        p = f"backbone.layers.{i}."
        for mine, theirs in (("norm1", "ln1"), ("norm2", "ln2"), ("attn.qkv", "attn.qkv"), ("attn.proj", "attn.proj"),
                             ("fc1", "ffn.layers.0.0"), ("fc2", "ffn.layers.1")):
            out[f"blocks.{i}.{mine}.weight"] = src[p + theirs + ".weight"]
            out[f"blocks.{i}.{mine}.bias"] = src[p + theirs + ".bias"]
    out.update(_head_from(src, {"deconv1": "head.deconv_layers.0", "deconv2": "head.deconv_layers.3",
                                "bn1": "head.deconv_layers.1", "bn2": "head.deconv_layers.4", "final": "head.final_layer"}))
    return out


def coco_head_to_pt18(state: dict) -> dict:
    """Re-order the final 1x1 conv from COCO-17 channels to PoseTrack18-17 slots."""
    idx = torch.tensor(PT18_FROM_COCO)
    state = dict(state)
    state["final.weight"] = state["final.weight"][idx].clone()
    state["final.bias"] = state["final.bias"][idx].clone()
    return state


def resize_pos_embed(pos_embed: torch.Tensor, grid: tuple[int, int]) -> torch.Tensor:
    """Bicubic resize of the patch position embeddings (keeps the extra first token)."""
    cls, patches = pos_embed[:, :1], pos_embed[:, 1:]
    n = patches.shape[1]
    if n == grid[0] * grid[1]:
        return pos_embed
    old = int(round(math.sqrt(n * 4 / 3))), int(round(math.sqrt(n * 3 / 4)))  # 256x192 grids are 4:3
    if old[0] * old[1] != n:
        raise ValueError(f"cannot infer pos_embed grid from {n} tokens")
    patches = patches.reshape(1, old[0], old[1], -1).permute(0, 3, 1, 2)
    patches = F.interpolate(patches, size=grid, mode="bicubic", align_corners=False)
    return torch.cat([cls, patches.permute(0, 2, 3, 1).reshape(1, grid[0] * grid[1], -1)], dim=1)


def load_checkpoint_file(path: str) -> dict:
    """Load a .pth checkpoint without executing arbitrary pickle code.

    MMPose checkpoints (legacy torch format) pickle a few numpy objects in their
    metadata, which torch's weights_only loader rejects under numpy 2. A restricted
    unpickler admits exactly the tensor-rebuild, OrderedDict and numpy array/dtype
    globals and refuses everything else."""
    import collections
    import importlib
    import pickle
    import types

    allowed_modules = {"numpy.core.multiarray": "_reconstruct", "numpy._core.multiarray": "_reconstruct"}
    allowed = {("collections", "OrderedDict"), ("torch._utils", "_rebuild_tensor_v2"),
               ("torch._utils", "_rebuild_parameter"), ("numpy", "ndarray"), ("numpy", "dtype"),
               ("_codecs", "encode")}

    class SafeUnpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if module == "torch" and name.endswith("Storage"):
                return getattr(importlib.import_module("torch"), name)
            if module == "numpy.dtypes" and name.endswith("DType"):
                return getattr(importlib.import_module("numpy.dtypes"), name)
            if allowed_modules.get(module) == name or (module, name) in allowed:
                if module.startswith("numpy") and "core" in module:
                    try:
                        return importlib.import_module("numpy._core.multiarray")._reconstruct
                    except ImportError:
                        return importlib.import_module("numpy.core.multiarray")._reconstruct
                if module == "collections":
                    return collections.OrderedDict
                return getattr(importlib.import_module(module), name)
            raise pickle.UnpicklingError(f"refusing to unpickle {module}.{name}")

    safe_pickle = types.SimpleNamespace(
        __name__="safe_pickle", Unpickler=SafeUnpickler,
        load=lambda f, **kw: SafeUnpickler(f, **kw).load())
    return torch.load(path, map_location="cpu", pickle_module=safe_pickle, weights_only=False)

In [ ]:
%%writefile /content/upar/pose_infer.py
"""Top-down heatmap pose inference shared by the Colab experiments and run.py.

numpy only (plus a model callable), so it runs unchanged in the evaluation
image (torch 2.4.1, numpy, Pillow; no OpenCV/scipy). Preprocessing and DARK
decoding follow the organizers' example submission; the Gaussian blur is
vectorised and flip test-time augmentation is added.

The model is passed in as `forward(batch) -> heatmaps`, with batch a float32
array (B, 3, H, W) and heatmaps a float32 array (B, K, h, w).
"""
from __future__ import annotations

import numpy as np

# COCO-17 left/right pairs (nose=0, eyes 1/2, ears 3/4, shoulders 5/6, ...).
COCO_FLIP_PAIRS = ((1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16))
# PoseTrack18-17: nose, head_bottom and head_top are centre joints.
PT18_FLIP_PAIRS = ((3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16))
# COCO joints whose positions predict head_bottom / head_top.
HEAD_FEATURE_JOINTS = (0, 1, 2, 3, 4, 5, 6)


# ---------------------------------------------------------------- preprocessing

def box_to_center_scale(box, out_w: int, out_h: int, normalize_factor: float = 200.0,
                        padding_factor: float = 1.25):
    x, y, w, h = [float(v) for v in box[:4]]
    aspect = out_w / out_h
    center = np.array([x + w * 0.5, y + h * 0.5], dtype=np.float32)
    if w > aspect * h:
        h = w / aspect
    elif w < aspect * h:
        w = h * aspect
    scale = np.array([w / normalize_factor, h / normalize_factor], dtype=np.float32)
    return center, scale * padding_factor


def warp_matrix(center: np.ndarray, scale: np.ndarray, out_w: int, out_h: int) -> np.ndarray:
    size_input = center * 2.0
    size_dst = np.array([out_w, out_h], dtype=np.float32) - 1.0
    size_target = scale * 200.0
    scale_x = size_dst[0] / size_target[0]
    scale_y = size_dst[1] / size_target[1]
    matrix = np.zeros((2, 3), dtype=np.float32)
    matrix[0, 0] = scale_x
    matrix[0, 2] = scale_x * (-0.5 * size_input[0] + 0.5 * size_target[0])
    matrix[1, 1] = scale_y
    matrix[1, 2] = scale_y * (-0.5 * size_input[1] + 0.5 * size_target[1])
    return matrix


def warp_affine(image: np.ndarray, matrix: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    """Bilinear warp with zero fill outside the image (numpy stand-in for cv2.warpAffine)."""
    bottom = np.array([[0.0, 0.0, 1.0]], dtype=np.float32)
    inverse = np.linalg.inv(np.vstack([matrix, bottom]))[:2].astype(np.float32)
    ys, xs = np.meshgrid(np.arange(out_h, dtype=np.float32), np.arange(out_w, dtype=np.float32), indexing="ij")
    src_x = inverse[0, 0] * xs + inverse[0, 1] * ys + inverse[0, 2]
    src_y = inverse[1, 0] * xs + inverse[1, 1] * ys + inverse[1, 2]

    height, width = image.shape[:2]
    x0 = np.floor(src_x).astype(np.int64)
    y0 = np.floor(src_y).astype(np.int64)
    x1, y1 = x0 + 1, y0 + 1
    wx, wy = src_x - x0, src_y - y0
    inside = (src_x >= 0) & (src_x <= width - 1) & (src_y >= 0) & (src_y <= height - 1)
    x0c, x1c = np.clip(x0, 0, width - 1), np.clip(x1, 0, width - 1)
    y0c, y1c = np.clip(y0, 0, height - 1), np.clip(y1, 0, height - 1)
    top = image[y0c, x0c] * (1 - wx)[..., None] + image[y0c, x1c] * wx[..., None]
    bot = image[y1c, x0c] * (1 - wx)[..., None] + image[y1c, x1c] * wx[..., None]
    return (top * (1 - wy)[..., None] + bot * wy[..., None]) * inside[..., None]


def preprocess(image: np.ndarray, box, cfg: dict):
    """image: HxWx3 float32 RGB in [0, 255]. Returns (CHW tensor, center, scale)."""
    out_w, out_h = int(cfg["input_width"]), int(cfg["input_height"])
    center, scale = box_to_center_scale(box, out_w, out_h, float(cfg.get("normalize_factor", 200.0)),
                                        float(cfg.get("padding_factor", 1.25)))
    crop = warp_affine(image, warp_matrix(center, scale, out_w, out_h), out_h, out_w)
    mean = np.asarray(cfg["image_mean"], dtype=np.float32).reshape(3, 1, 1)
    std = np.asarray(cfg["image_std"], dtype=np.float32).reshape(3, 1, 1)
    tensor = crop.transpose(2, 0, 1) * float(cfg.get("rescale_factor", 1.0 / 255.0))
    return ((tensor - mean) / std).astype(np.float32), center, scale


# ---------------------------------------------------------------- decoding

def _gaussian_kernel1d(sigma: float, radius: int) -> np.ndarray:
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    kernel = np.exp(-0.5 / (sigma * sigma) * x * x)
    return (kernel / kernel.sum()).astype(np.float32)


def gaussian_blur(heatmaps: np.ndarray, kernel_size: int = 11, sigma: float = 0.8) -> np.ndarray:
    """Separable blur over the last two axes with symmetric padding (vectorised)."""
    radius = (kernel_size - 1) // 2
    kernel = _gaussian_kernel1d(sigma, radius)
    h, w = heatmaps.shape[-2:]
    padded = np.pad(heatmaps, ((0, 0), (0, 0), (radius, radius), (radius, radius)), mode="symmetric")
    rows = sum(kernel[k] * padded[..., :, k:k + w] for k in range(kernel_size))
    return sum(kernel[k] * rows[..., k:k + h, :] for k in range(kernel_size))


def decode_heatmaps(heatmaps: np.ndarray, centers: np.ndarray, scales: np.ndarray, dark_kernel: int = 11):
    """Argmax + DARK refinement. Returns image coords (B, K, 2) and max values (B, K)."""
    batch, num_kpts, height, width = heatmaps.shape
    flat = heatmaps.reshape(batch, num_kpts, -1)
    idx = np.argmax(flat, axis=2)
    scores = np.max(flat, axis=2)
    coords = np.zeros((batch, num_kpts, 2), dtype=np.float32)
    coords[:, :, 0] = idx % width
    coords[:, :, 1] = idx // width

    blurred = np.clip(gaussian_blur(heatmaps, dark_kernel), 0.001, 50.0)
    padded = np.pad(np.log(blurred), ((0, 0), (0, 0), (1, 1), (1, 1)), mode="edge").reshape(-1)
    stride = (width + 2) * (height + 2)
    base = coords[..., 0] + 1 + (coords[..., 1] + 1) * (width + 2)
    base = base + stride * np.arange(batch * num_kpts).reshape(batch, num_kpts)
    index = base.astype(np.int64).reshape(-1, 1)

    center_v = padded[index]
    x_plus, x_minus = padded[index + 1], padded[index - 1]
    y_plus, y_minus = padded[index + width + 2], padded[index - 2 - width]
    xy_plus, xy_minus = padded[index + width + 3], padded[index - width - 3]
    dx = 0.5 * (x_plus - x_minus)
    dy = 0.5 * (y_plus - y_minus)
    derivative = np.concatenate([dx, dy], axis=1).reshape(batch, num_kpts, 2, 1)
    dxx = x_plus - 2 * center_v + x_minus
    dyy = y_plus - 2 * center_v + y_minus
    dxy = 0.5 * (xy_plus - x_plus - y_plus + center_v + center_v - x_minus - y_minus + xy_minus)
    hessian = np.concatenate([dxx, dxy, dxy, dyy], axis=1).reshape(batch, num_kpts, 2, 2)
    hessian = np.linalg.inv(hessian + np.finfo(np.float32).eps * np.eye(2))
    coords = coords - np.einsum("ijmn,ijnk->ijmk", hessian, derivative).squeeze(-1)

    out = np.zeros_like(coords)
    for i in range(batch):
        size = scales[i] * 200.0
        out[i, :, 0] = coords[i, :, 0] * size[0] / (width - 1.0) + centers[i][0] - size[0] * 0.5
        out[i, :, 1] = coords[i, :, 1] * size[1] / (height - 1.0) + centers[i][1] - size[1] * 0.5
    return out, scores


def flip_back(heatmaps: np.ndarray, flip_pairs, shift: bool = False) -> np.ndarray:
    """Undo a horizontal input flip on heatmaps: mirror and swap left/right channels.
    ViTPose uses UDP, so no one-pixel shift (MMPose: shift_heatmap=False); `shift=True`
    is only for non-UDP models. Measured on the mock test: no shift 0.8866, shift 0.8700."""
    out = heatmaps[..., ::-1].copy()
    for a, b in flip_pairs:
        out[:, [a, b]] = out[:, [b, a]]
    if shift:
        out[..., 1:] = out[..., :-1].copy()
    return out


def predict_boxes(image: np.ndarray, boxes, forward, cfg: dict, flip: bool = False,
                  flip_pairs=COCO_FLIP_PAIRS, batch_size: int = 16, flip_shift: bool = False):
    """Run the model on every usable box. Returns (keypoints (N, K, 2), joint_scores (N, K),
    usable mask (N,)). Boxes with w or h <= 0 are left as NaN for the caller to fill."""
    num_kpts = int(cfg["num_keypoints"])
    n = len(boxes)
    keypoints = np.full((n, num_kpts, 2), np.nan, dtype=np.float32)
    joint_scores = np.zeros((n, num_kpts), dtype=np.float32)
    usable = np.array([float(b[2]) > 0 and float(b[3]) > 0 for b in boxes], dtype=bool)
    order = np.flatnonzero(usable)
    for start in range(0, len(order), batch_size):
        chunk = order[start:start + batch_size]
        tensors, centers, scales = zip(*(preprocess(image, boxes[i], cfg) for i in chunk))
        batch = np.stack(tensors)
        heatmaps = forward(batch)
        if flip:
            heatmaps = 0.5 * (heatmaps + flip_back(forward(np.ascontiguousarray(batch[..., ::-1])), flip_pairs, flip_shift))
        coords, maxvals = decode_heatmaps(heatmaps, np.stack(centers), np.stack(scales))
        keypoints[chunk] = coords
        joint_scores[chunk] = maxvals
    return keypoints, joint_scores, usable



def _corner_aligned_matrix(n_out: int, n_in: int) -> np.ndarray:
    pos = np.arange(n_out, dtype=np.float64) * (n_in - 1) / max(n_out - 1, 1)
    lo = np.clip(np.floor(pos).astype(np.int64), 0, n_in - 2)
    frac = (pos - lo).astype(np.float32)
    m = np.zeros((n_out, n_in), dtype=np.float32)
    m[np.arange(n_out), lo] = 1.0 - frac
    m[np.arange(n_out), lo + 1] += frac
    return m


def resize_heatmaps(heatmaps: np.ndarray, height: int, width: int) -> np.ndarray:
    """Bilinear resize with aligned corners. decode_heatmaps maps heatmap pixel 0 and w-1 to
    the crop edges, so heatmaps of the same crop at different resolutions (e.g. 64x48 from a
    256x192 model, 96x72 from a 384x288 one) line up after this resize."""
    h, w = heatmaps.shape[-2:]
    if (h, w) == (height, width):
        return heatmaps
    return np.einsum("yi,bkij,xj->bkyx", _corner_aligned_matrix(height, h), heatmaps,
                     _corner_aligned_matrix(width, w), optimize=True).astype(np.float32)


def predict_boxes_ensemble(image: np.ndarray, boxes, members, flip: bool = False,
                           flip_pairs=COCO_FLIP_PAIRS, batch_size: int = 16, flip_shift: bool = False):
    """predict_boxes for a heatmap ensemble. members: list of (forward, cfg, weight); the configs
    may differ in input size but must share aspect ratio and padding, so every member sees the
    same crop. Each member's (flip-averaged) heatmaps are resized to the largest heatmap size,
    weighted, summed and decoded once."""
    cfgs = [cfg for _, cfg, _ in members]
    num_kpts = int(cfgs[0]["num_keypoints"])
    n = len(boxes)
    keypoints = np.full((n, num_kpts, 2), np.nan, dtype=np.float32)
    joint_scores = np.zeros((n, num_kpts), dtype=np.float32)
    usable = np.array([float(b[2]) > 0 and float(b[3]) > 0 for b in boxes], dtype=bool)
    order = np.flatnonzero(usable)
    out_h = max(int(c["input_height"]) // 4 for c in cfgs)
    out_w = max(int(c["input_width"]) // 4 for c in cfgs)
    for start in range(0, len(order), batch_size):
        chunk = order[start:start + batch_size]
        total, centers, scales = None, None, None
        for forward, cfg, weight in members:
            tensors, c, s = zip(*(preprocess(image, boxes[i], cfg) for i in chunk))
            if centers is None:
                centers, scales = np.stack(c), np.stack(s)
            elif not (np.allclose(centers, np.stack(c)) and np.allclose(scales, np.stack(s))):
                raise ValueError("ensemble members crop different regions (aspect ratio / padding differ)")
            batch = np.stack(tensors)
            heatmaps = forward(batch)
            if flip:
                heatmaps = 0.5 * (heatmaps + flip_back(forward(np.ascontiguousarray(batch[..., ::-1])), flip_pairs, flip_shift))
            heatmaps = float(weight) * resize_heatmaps(heatmaps, out_h, out_w)
            total = heatmaps if total is None else total + heatmaps
        coords, maxvals = decode_heatmaps(total, centers, scales)
        keypoints[chunk] = coords
        joint_scores[chunk] = maxvals
    return keypoints, joint_scores, usable

# ---------------------------------------------------------------- COCO -> PoseTrack18

def head_features(coco_kpts: np.ndarray, box) -> np.ndarray:
    """Box-normalised (u, v) of the head/shoulder COCO joints + bias. coco_kpts: (17, 2)."""
    x, y, w, h = [float(v) for v in box[:4]]
    pts = coco_kpts[list(HEAD_FEATURE_JOINTS)]
    uv = np.stack([(pts[:, 0] - x) / w, (pts[:, 1] - y) / h], axis=1).reshape(-1)
    return np.concatenate([uv, [1.0]]).astype(np.float64)


def coco_to_pt18(coco_kpts: np.ndarray, coco_scores: np.ndarray, box, mapping: dict | None):
    """Map a COCO-17 prediction to the PoseTrack18 slot order. Slots 1/2 (eyes in COCO)
    become head_bottom/head_top predicted by a linear model on head/shoulder joints.
    Without a mapping the eyes are copied through (the organizers' baseline behaviour)."""
    kpts = coco_kpts.astype(np.float64).copy()
    scores = coco_scores.astype(np.float64).copy()
    if mapping is not None:
        x, y, w, h = [float(v) for v in box[:4]]
        f = head_features(coco_kpts, box)
        feature_score = float(np.mean(coco_scores[list(HEAD_FEATURE_JOINTS)]))
        for slot, name in ((1, "head_bottom"), (2, "head_top")):
            coef_u, coef_v = mapping[name]
            kpts[slot] = [x + float(f @ np.asarray(coef_u)) * w, y + float(f @ np.asarray(coef_v)) * h]
            scores[slot] = feature_score
    return kpts, scores


OKS_SIGMAS = np.array([
    0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072,
    0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089
], dtype=np.float64)
SCORED_JOINTS = [0, 1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
HEAD_JOINTS = [0, 1, 2]
INV_SIGMA = 1.0 / OKS_SIGMAS[SCORED_JOINTS]
WEIGHTS_SIGMA = INV_SIGMA / INV_SIGMA.sum()


def person_score(joint_scores: np.ndarray, mode: str = "mean_scored", threshold: float = 0.2,
                 ignore=(3, 4), gamma: float = 0.15) -> float:
    """Per-person confidence used to rank detections in OKS-AP."""
    s = np.asarray(joint_scores, dtype=np.float64)
    if mode == "mean_all":
        return float(s.mean())
    if mode == "calibrated_oks":
        scored_scores = s[SCORED_JOINTS]
        weighted_mean = float(np.sum(scored_scores * WEIGHTS_SIGMA))
        if gamma > 0.0:
            min_head = float(np.clip(np.min(s[HEAD_JOINTS]), 1e-4, 1.0))
            return weighted_mean * (min_head ** gamma)
        return weighted_mean
    keep = np.ones(len(s), dtype=bool)
    keep[list(ignore)] = False
    s = s[keep]
    if mode == "mean_scored":
        return float(s.mean())
    if mode == "mean_thresh":
        good = s[s > threshold]
        return float(good.mean()) if len(good) else float(s.mean())
    raise ValueError(f"unknown score mode {mode}")

In [ ]:
%%writefile /content/upar/coco_kpt_eval.py
"""Dependency-light COCO keypoint evaluation (OKS-AP), numpy only.

Reimplements pycocotools.cocoeval.COCOeval(iouType='keypoints') for a single
category, following the organizers' description: AP over OKS 0.50:0.95,
maxDets = 20, areas all / medium / large. Ground truth with no labelled
keypoints and crowd regions are ignored. Joints with v=0 in the ground truth
(e.g. the unannotated ear slots) do not enter the OKS.

The per-joint OKS sigmas used by the challenge are not published. The
default below is the COCO set, which is also what mmpose uses for
PoseTrack18 (head_bottom / head_top get the eye sigmas). Override with
--sigmas or a "sigmas" entry in the task file if the organizers publish theirs.

Usage:
  python -I coco_kpt_eval.py --gt ground_truth.json --dt predictions.json
"""
from __future__ import annotations

import argparse
import json
from collections import defaultdict

import numpy as np

DEFAULT_SIGMAS = np.array([
    0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072,
    0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089,
])
IOU_THRS = np.linspace(0.5, 0.95, 10)
REC_THRS = np.linspace(0.0, 1.0, 101)
MAX_DETS = 20
AREA_RNG = {"all": (0.0, 1e10), "medium": (32.0 ** 2, 96.0 ** 2), "large": (96.0 ** 2, 1e10)}


def _dt_area_from_keypoints(kpts: np.ndarray) -> float:
    # loadRes() derives the detection area from the keypoint extent.
    xs, ys = kpts[0::3], kpts[1::3]
    return float((xs.max() - xs.min()) * (ys.max() - ys.min()))


def compute_oks(gts: list[dict], dts: list[dict], sigmas: np.ndarray) -> np.ndarray:
    if not gts or not dts:
        return np.zeros((len(dts), len(gts)))
    variances = (sigmas * 2) ** 2
    ious = np.zeros((len(dts), len(gts)))
    for j, gt in enumerate(gts):
        g = np.asarray(gt["keypoints"], dtype=np.float64)
        xg, yg, vg = g[0::3], g[1::3], g[2::3]
        k1 = np.count_nonzero(vg > 0)
        bx, by, bw, bh = gt["bbox"]
        x0, x1, y0, y1 = bx - bw, bx + bw * 2, by - bh, by + bh * 2
        for i, dt in enumerate(dts):
            d = np.asarray(dt["keypoints"], dtype=np.float64)
            xd, yd = d[0::3], d[1::3]
            if k1 > 0:
                dx, dy = xd - xg, yd - yg
            else:
                z = np.zeros(len(sigmas))
                dx = np.maximum(z, x0 - xd) + np.maximum(z, xd - x1)
                dy = np.maximum(z, y0 - yd) + np.maximum(z, yd - y1)
            e = (dx ** 2 + dy ** 2) / variances / (gt["area"] + np.spacing(1)) / 2
            if k1 > 0:
                e = e[vg > 0]
            ious[i, j] = np.sum(np.exp(-e)) / e.shape[0]
    return ious


def _evaluate_image(gts, dts, ious, area_rng):
    """Returns (dt_scores, dt_matched[T,D], dt_ignore[T,D], num_gt_not_ignored) or None."""
    if not gts and not dts:
        return None
    gt_ignore = np.array([1 if (g["_ignore"] or g["area"] < area_rng[0] or g["area"] > area_rng[1]) else 0
                          for g in gts], dtype=int)
    gt_order = np.argsort(gt_ignore, kind="mergesort")
    gts = [gts[i] for i in gt_order]
    gt_ignore = gt_ignore[gt_order]
    is_crowd = np.array([g.get("iscrowd", 0) for g in gts], dtype=int)

    dt_order = np.argsort([-d["score"] for d in dts], kind="mergesort")[:MAX_DETS]
    dts = [dts[i] for i in dt_order]
    ious = ious[dt_order][:, gt_order] if len(ious) else ious

    T, G, D = len(IOU_THRS), len(gts), len(dts)
    gt_matched = np.zeros((T, G))
    dt_matched = np.zeros((T, D))
    dt_ignore = np.zeros((T, D))
    if len(ious):
        for t_index, t in enumerate(IOU_THRS):
            for d_index in range(D):
                best = min(t, 1 - 1e-10)
                m = -1
                for g_index in range(G):
                    if gt_matched[t_index, g_index] > 0 and not is_crowd[g_index]:
                        continue
                    if m > -1 and gt_ignore[m] == 0 and gt_ignore[g_index] == 1:
                        break
                    if ious[d_index, g_index] < best:
                        continue
                    best = ious[d_index, g_index]
                    m = g_index
                if m == -1:
                    continue
                dt_ignore[t_index, d_index] = gt_ignore[m]
                dt_matched[t_index, d_index] = 1
                gt_matched[t_index, m] = 1
    dt_out_of_range = np.array([d["area"] < area_rng[0] or d["area"] > area_rng[1] for d in dts]).reshape(1, D)
    dt_ignore = np.logical_or(dt_ignore, np.logical_and(dt_matched == 0, np.repeat(dt_out_of_range, T, 0)))
    scores = np.array([d["score"] for d in dts])
    return scores, dt_matched, dt_ignore, int(np.count_nonzero(gt_ignore == 0))


def _accumulate(per_image):
    """AP and AR at each IoU threshold for one area range."""
    per_image = [r for r in per_image if r is not None]
    T = len(IOU_THRS)
    if not per_image:
        return np.full(T, -1.0), np.full(T, -1.0)
    scores = np.concatenate([r[0] for r in per_image])
    order = np.argsort(-scores, kind="mergesort")
    dtm = np.concatenate([r[1] for r in per_image], axis=1)[:, order]
    dtig = np.concatenate([r[2] for r in per_image], axis=1)[:, order]
    npig = sum(r[3] for r in per_image)
    if npig == 0:
        return np.full(T, -1.0), np.full(T, -1.0)

    tps = np.logical_and(dtm, np.logical_not(dtig))
    fps = np.logical_and(np.logical_not(dtm), np.logical_not(dtig))
    tp_sum = np.cumsum(tps, axis=1).astype(float)
    fp_sum = np.cumsum(fps, axis=1).astype(float)
    ap = np.zeros(T)
    ar = np.zeros(T)
    for t in range(T):
        tp, fp = tp_sum[t], fp_sum[t]
        nd = len(tp)
        rc = tp / npig
        pr = tp / (fp + tp + np.spacing(1))
        ar[t] = rc[-1] if nd else 0.0
        pr = pr.tolist()
        for i in range(nd - 1, 0, -1):
            if pr[i] > pr[i - 1]:
                pr[i - 1] = pr[i]
        inds = np.searchsorted(rc, REC_THRS, side="left")
        q = np.zeros(len(REC_THRS))
        for ri, pi in enumerate(inds):
            if pi >= nd:
                break
            q[ri] = pr[pi]
        ap[t] = q.mean()
    return ap, ar


def evaluate(gt: dict, predictions: list[dict], sigmas: np.ndarray = DEFAULT_SIGMAS) -> dict:
    """gt: COCO dict with keypoints. predictions: COCO results list."""
    gts_by_image = defaultdict(list)
    for ann in gt["annotations"]:
        ann = dict(ann)
        num_kpts = ann.get("num_keypoints", int(np.count_nonzero(np.asarray(ann["keypoints"])[2::3] > 0)))
        ann["_ignore"] = bool(ann.get("iscrowd", 0)) or num_kpts == 0
        gts_by_image[ann["image_id"]].append(ann)
    dts_by_image = defaultdict(list)
    for det in predictions:
        det = dict(det)
        det["area"] = _dt_area_from_keypoints(np.asarray(det["keypoints"], dtype=np.float64))
        dts_by_image[det["image_id"]].append(det)

    image_ids = sorted(img["id"] for img in gt["images"])
    ious = {i: compute_oks(gts_by_image[i], sorted(dts_by_image[i], key=lambda d: -d["score"])[:MAX_DETS], sigmas)
            for i in image_ids}
    # compute_oks above already sees score-sorted detections; keep the same order for matching.
    for i in image_ids:
        dts_by_image[i] = sorted(dts_by_image[i], key=lambda d: -d["score"])[:MAX_DETS]

    results = {}
    for name, rng in AREA_RNG.items():
        per_image = [_evaluate_image(gts_by_image[i], dts_by_image[i], ious[i], rng) for i in image_ids]
        ap, ar = _accumulate(per_image)
        results[name] = (ap, ar)

    def mean_valid(x):
        x = x[x > -1]
        return float(x.mean()) if len(x) else -1.0

    ap_all, ar_all = results["all"]
    return {
        "AP": mean_valid(ap_all),
        "AP50": float(ap_all[0]),
        "AP75": float(ap_all[5]),
        "AP_medium": mean_valid(results["medium"][0]),
        "AP_large": mean_valid(results["large"][0]),
        "AR": mean_valid(ar_all),
        "AR50": float(ar_all[0]),
        "AR75": float(ar_all[5]),
        "AR_medium": mean_valid(results["medium"][1]),
        "AR_large": mean_valid(results["large"][1]),
    }


def per_keypoint_similarity(gt: dict, predictions: list[dict], sigmas: np.ndarray = DEFAULT_SIGMAS) -> dict:
    """Diagnostic: mean per-joint OKS term exp(-e) and PCK@0.5 over labelled joints,
    pairing each prediction with the ground truth box it was made for (box_id)."""
    names = gt["categories"][0]["keypoints"]
    variances = (sigmas * 2) ** 2
    anns = {a["id"]: a for a in gt["annotations"]}
    sums, hits, counts = np.zeros(len(names)), np.zeros(len(names)), np.zeros(len(names))
    for det in predictions:
        ann = anns.get(det.get("box_id"))
        if ann is None:
            continue
        g = np.asarray(ann["keypoints"], dtype=np.float64)
        d = np.asarray(det["keypoints"], dtype=np.float64)
        vis = g[2::3] > 0
        e = ((d[0::3] - g[0::3]) ** 2 + (d[1::3] - g[1::3]) ** 2) / variances / (ann["area"] + np.spacing(1)) / 2
        ks = np.exp(-e)
        sums[vis] += ks[vis]
        hits[vis] += ks[vis] > 0.5
        counts[vis] += 1
    out = {}
    for j, name in enumerate(names):
        if counts[j]:
            out[name] = {"oks": float(sums[j] / counts[j]), "pck50": float(hits[j] / counts[j]), "n": int(counts[j])}
    return out


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--gt", required=True)
    parser.add_argument("--dt", required=True)
    parser.add_argument("--sigmas", type=float, nargs=17, default=None)
    parser.add_argument("--per-keypoint", action="store_true")
    args = parser.parse_args()

    with open(args.gt, "r", encoding="utf-8") as fh:
        gt = json.load(fh)
    with open(args.dt, "r", encoding="utf-8") as fh:
        dt = json.load(fh)
    sigmas = np.asarray(args.sigmas) if args.sigmas else DEFAULT_SIGMAS
    metrics = evaluate(gt, dt, sigmas)
    print(json.dumps(metrics, indent=2))
    if args.per_keypoint:
        for name, row in per_keypoint_similarity(gt, dt, sigmas).items():
            print(f"  {name:15s} oks={row['oks']:.3f} pck50={row['pck50']:.3f} n={row['n']}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/upar/build_mock_test.py
"""Build a local mock of the hidden test sets from a validation split.

Each subset mirrors the documented layout:
  <subset>/input_data/images/...    what the container sees (read-only)
  <subset>/input_data/boxes.json    COCO images + person boxes, no keypoints
  <subset>/input_data/task.json     set metadata
  <subset>/ground_truth.json        COCO with keypoints, never mounted

Four subsets, like the four hidden ones:
  crops_market, crops_pa100k, crops_peta  validation crops as they are
  frames_mixed                            validation crops pasted at random
                                          scales into larger frames, several
                                          people per frame, so there is context
                                          around each box and coordinates are
                                          absolute in a big image

The frames are only a rough stand-in for real full-frame surveillance images:
the background is a heavily blurred validation image, not a real scene.

Usage:
  python -I build_mock_test.py --data-dir <starter-kit>/data \
      --ann <prepared>/train_upar_all_fixed.json --splits <prepared>/splits.json \
      --split random5 --out-dir <mock_test>
"""
from __future__ import annotations

import argparse
import json
import random
import shutil
from pathlib import Path, PurePosixPath

from PIL import Image, ImageFilter

SOURCES = ("market", "pa100k", "peta")
FRAME_SIZES = ((640, 480), (960, 540), (1280, 720))
PEOPLE_PER_FRAME = (2, 6)
PASTE_SCALE = (0.75, 2.0)
MAX_OVERLAP = 0.1


def write_subset(out_dir: Path, name: str, description: str, images: list[dict], annotations: list[dict],
                 category: dict) -> None:
    subset_dir = out_dir / name
    input_dir = subset_dir / "input_data"
    input_dir.mkdir(parents=True, exist_ok=True)
    category = {k: category[k] for k in ("id", "name", "supercategory", "keypoints", "skeleton") if k in category}

    gt = {"images": images, "annotations": annotations, "categories": [category]}
    boxes = {
        "images": images,
        "annotations": [{k: v for k, v in a.items() if k not in ("keypoints", "num_keypoints")} for a in annotations],
        "categories": [category],
    }
    task = {
        "name": name,
        "title": f"Local mock: {name}",
        "description": description,
        "protocol": "COCO keypoints OKS-AP, maxDets=20 (local mock)",
        "size": len(images),
        "keypoint_names": category["keypoints"],
        "skeleton": category["skeleton"],
        "score_prefix": name,
    }
    (subset_dir / "ground_truth.json").write_text(json.dumps(gt), encoding="utf-8")
    (input_dir / "boxes.json").write_text(json.dumps(boxes), encoding="utf-8")
    (input_dir / "task.json").write_text(json.dumps(task, indent=2), encoding="utf-8")
    print(f"[mock] {name:13s} images={len(images):5d} boxes={len(annotations):5d}")


def build_crops(data_dir: Path, out_dir: Path, source: str, images: list[dict], anns_by_image: dict,
                category: dict) -> None:
    name = f"crops_{source}"
    image_dir = out_dir / name / "input_data" / "images"
    image_dir.mkdir(parents=True, exist_ok=True)
    out_images, out_anns = [], []
    for img in images:
        file_name = f"{img['id']:06d}{PurePosixPath(img['file_name']).suffix}"
        shutil.copyfile(data_dir / img["file_name"], image_dir / file_name)
        out_images.append({"id": img["id"], "file_name": file_name, "width": img["width"], "height": img["height"]})
        for ann in anns_by_image.get(img["id"], []):
            out_anns.append({"id": ann["id"], "image_id": img["id"], "category_id": 1, "bbox": ann["bbox"],
                             "area": ann["area"], "iscrowd": ann.get("iscrowd", 0),
                             "keypoints": ann["keypoints"], "num_keypoints": ann["num_keypoints"]})
    write_subset(out_dir, name, f"Validation crops from {source}, one person per image.",
                 out_images, out_anns, category)


def _overlap(a, b) -> float:
    ix = max(0.0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / min(a[2] * a[3], b[2] * b[3])


def build_frames(data_dir: Path, out_dir: Path, images: list[dict], anns_by_image: dict, category: dict,
                 rng: random.Random) -> None:
    name = "frames_mixed"
    image_dir = out_dir / name / "input_data" / "images"
    image_dir.mkdir(parents=True, exist_ok=True)
    pool = list(images)
    rng.shuffle(pool)
    out_images, out_anns = [], []
    frame_id = 0
    while pool:
        frame_id += 1
        width, height = rng.choice(FRAME_SIZES)
        bg_src = Image.open(data_dir / rng.choice(images)["file_name"]).convert("RGB")
        frame = bg_src.resize((width, height), Image.BILINEAR).filter(ImageFilter.GaussianBlur(40))

        placed = []
        for _ in range(rng.randint(*PEOPLE_PER_FRAME)):
            if not pool:
                break
            img = pool[-1]
            scale = rng.uniform(*PASTE_SCALE)
            w, h = round(img["width"] * scale), round(img["height"] * scale)
            if w >= width or h >= height:
                scale = min((width - 1) / img["width"], (height - 1) / img["height"])
                w, h = round(img["width"] * scale), round(img["height"] * scale)
            for _attempt in range(20):
                x, y = rng.randint(0, width - w), rng.randint(0, height - h)
                if all(_overlap((x, y, w, h), p) <= MAX_OVERLAP for p in placed):
                    break
            else:
                break
            pool.pop()
            placed.append((x, y, w, h))
            crop = Image.open(data_dir / img["file_name"]).convert("RGB").resize((w, h), Image.BILINEAR)
            frame.paste(crop, (x, y))
            sx, sy = w / img["width"], h / img["height"]
            for ann in anns_by_image.get(img["id"], []):
                k = list(ann["keypoints"])
                for j in range(0, len(k), 3):
                    if k[j + 2] > 0:
                        k[j], k[j + 1] = k[j] * sx + x, k[j + 1] * sy + y
                bx, by, bw, bh = ann["bbox"]
                out_anns.append({"id": ann["id"], "image_id": frame_id, "category_id": 1,
                                 "bbox": [bx * sx + x, by * sy + y, bw * sx, bh * sy],
                                 "area": ann["area"] * sx * sy, "iscrowd": ann.get("iscrowd", 0),
                                 "keypoints": k, "num_keypoints": ann["num_keypoints"]})
        if not placed:
            continue
        file_name = f"frame_{frame_id:05d}.jpg"
        frame.save(image_dir / file_name, quality=92)
        out_images.append({"id": frame_id, "file_name": file_name, "width": width, "height": height})

    # Pasted later people can occlude earlier ones; that is fine for a mock.
    write_subset(out_dir, name, "Validation crops pasted into larger frames (several people per frame).",
                 out_images, out_anns, category)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--ann", type=Path, required=True)
    parser.add_argument("--splits", type=Path, required=True)
    parser.add_argument("--split", default="random5")
    parser.add_argument("--out-dir", type=Path, required=True)
    parser.add_argument("--max-per-source", type=int, default=0, help="cap images per source (0 = all)")
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    data = json.loads(args.ann.read_text(encoding="utf-8"))
    val_ids = set(json.loads(args.splits.read_text(encoding="utf-8"))[args.split]["val"])
    category = data["categories"][0]
    anns_by_image = {}
    for ann in data["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    rng = random.Random(args.seed)
    out_dir = args.out_dir / args.split
    if out_dir.exists():
        shutil.rmtree(out_dir)
    val_images = [img for img in data["images"] if img["id"] in val_ids]

    used = []
    for source in SOURCES:
        images = [img for img in val_images if img["source"] == source]
        if args.max_per_source:
            images = rng.sample(images, min(args.max_per_source, len(images)))
        if not images:
            continue
        used.extend(images)
        build_crops(args.data_dir, out_dir, source, images, anns_by_image, category)
    build_frames(args.data_dir, out_dir, used, anns_by_image, category, rng)
    print(f"[mock] wrote {out_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/upar/data.py
"""Training data for top-down PoseTrack18 heatmap regression (runs on Colab).

Augmentation follows MMPose's ViTPose COCO recipe (random flip, half body,
bbox scale/shift/rotation, UDP affine, UDP Gaussian heatmaps sigma 2) plus the
organizers' cross-domain DA (Cormier, KIT 2026, Sec. 6.2): photometric
distortion and channel swap (p=0.5 each), Gaussian/median blur (p=0.1), one
coarse-dropout hole up to 40% of height/width. Grayscale and down/up-sampling
are added for unknown target domains.

Training uses OpenCV (available on Colab); the submission does not.
"""
from __future__ import annotations

import math

import cv2
import numpy as np
import torch

PT18_FLIP_PAIRS = ((3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16))
UPPER_BODY = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
IGNORED_SLOTS = (3, 4)  # ears: not annotated, not scored

DEFAULT_AUG = dict(
    flip=0.5, half_body=0.3, scale=(0.75, 1.25), shift=0.16, rotate=40.0, rotate_p=0.6,
    photometric=0.5, blur=0.1, median_blur=0.1, dropout=1.0, dropout_max=0.4,
    grayscale=0.1, lowres=0.2,
)


def udp_warp_matrix(center, size, rot_deg: float, out_w: int, out_h: int) -> np.ndarray:
    """MMPose get_udp_warp_matrix: maps the (rotated) box of `size` around `center`
    onto the (out_w-1) x (out_h-1) input grid."""
    theta = math.radians(rot_deg)
    sx = (out_w - 1.0) / size[0]
    sy = (out_h - 1.0) / size[1]
    c, s = math.cos(theta), math.sin(theta)
    return np.array([
        [c * sx, -s * sx, sx * (-center[0] * c + center[1] * s + 0.5 * size[0])],
        [s * sy, c * sy, sy * (-center[0] * s - center[1] * c + 0.5 * size[1])],
    ], dtype=np.float32)


def box_center_size(box, aspect: float, padding: float = 1.25):
    x, y, w, h = [float(v) for v in box[:4]]
    center = np.array([x + 0.5 * w, y + 0.5 * h], dtype=np.float32)
    if w > aspect * h:
        h = w / aspect
    else:
        w = h * aspect
    return center, np.array([w, h], dtype=np.float32) * padding


def udp_heatmaps(kpts_in: np.ndarray, weights: np.ndarray, in_size, hm_size, sigma: float = 2.0):
    """Gaussian targets at the exact (sub-pixel) UDP heatmap location."""
    w_hm, h_hm = hm_size
    feat = np.array([(w_hm - 1.0) / (in_size[0] - 1.0), (h_hm - 1.0) / (in_size[1] - 1.0)], dtype=np.float32)
    mu = kpts_in * feat
    xs = np.arange(w_hm, dtype=np.float32)[None, None, :]
    ys = np.arange(h_hm, dtype=np.float32)[None, :, None]
    hm = np.exp(-((xs - mu[:, 0, None, None]) ** 2 + (ys - mu[:, 1, None, None]) ** 2) / (2 * sigma ** 2))
    inside = (mu[:, 0] >= -0.5) & (mu[:, 0] <= w_hm - 0.5) & (mu[:, 1] >= -0.5) & (mu[:, 1] <= h_hm - 0.5)
    weights = weights * inside
    return (hm * weights[:, None, None]).astype(np.float32), weights.astype(np.float32)


def _photometric(img: np.ndarray, rng: np.random.Generator, p: float) -> np.ndarray:
    img = img.astype(np.float32)
    if rng.random() < p:
        img += rng.uniform(-32, 32)
    contrast_first = rng.random() < 0.5
    if contrast_first and rng.random() < p:
        img *= rng.uniform(0.5, 1.5)
    if rng.random() < p or rng.random() < p:
        hsv = cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        if rng.random() < p:
            hsv[..., 1] *= rng.uniform(0.5, 1.5)
        if rng.random() < p:
            hsv[..., 0] = (hsv[..., 0] + rng.uniform(-9, 9)) % 180  # OpenCV hue is 0..179 (= +/-18 deg)
        img = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)
    if not contrast_first and rng.random() < p:
        img *= rng.uniform(0.5, 1.5)
    if rng.random() < p:
        img = img[..., rng.permutation(3)]
    return np.clip(img, 0, 255).astype(np.uint8)


class PoseDataset(torch.utils.data.Dataset):
    """images: list of HxWx3 uint8 RGB arrays (kept in RAM, shared with fork workers).
    anns: list of dicts with 'image_index', 'bbox' and 'keypoints' (51 floats, PT18 order)."""

    def __init__(self, images, anns, cfg: dict, train: bool, aug: dict | None = None, seed: int = 0,
                 sigma: float = 2.0) -> None:
        self.images, self.anns, self.cfg, self.train = images, anns, cfg, train
        self.aug = {**DEFAULT_AUG, **(aug or {})}
        self.in_w, self.in_h = int(cfg["input_width"]), int(cfg["input_height"])
        self.hm_w, self.hm_h = self.in_w // 4, self.in_h // 4
        self.mean = np.asarray(cfg["image_mean"], dtype=np.float32) * 255.0
        self.std = np.asarray(cfg["image_std"], dtype=np.float32) * 255.0
        self.seed = seed
        self.sigma = sigma

    def __len__(self) -> int:
        return len(self.anns)

    def __getitem__(self, i: int):
        ann = self.anns[i]
        img = self.images[ann["image_index"]]
        kpts = np.asarray(ann["keypoints"], dtype=np.float32).reshape(-1, 3)
        vis = (kpts[:, 2] > 0).astype(np.float32)
        vis[list(IGNORED_SLOTS)] = 0.0
        aspect = self.in_w / self.in_h
        center, size = box_center_size(ann["bbox"], aspect)
        rot, flip = 0.0, False
        rng = np.random.default_rng() if self.train else None

        if self.train:
            a = self.aug
            if rng.random() < a["half_body"] and vis.sum() >= 8:
                part = [j for j in (UPPER_BODY if rng.random() < 0.5 else range(11, 17)) if vis[j] > 0]
                if len(part) >= 2:
                    pts = kpts[part, :2]
                    lo, hi = pts.min(0), pts.max(0)
                    center = (lo + hi) / 2
                    w, h = hi - lo
                    w, h = (h * aspect, h) if w < aspect * h else (w, w / aspect)
                    size = np.array([w, h], dtype=np.float32) * 1.5
            size = size * rng.uniform(*a["scale"])
            center = center + size * rng.uniform(-a["shift"], a["shift"], 2) * 0.5
            if rng.random() < a["rotate_p"]:
                rot = float(np.clip(rng.normal(0, a["rotate"] / 2), -a["rotate"], a["rotate"]))
            flip = rng.random() < a["flip"]

        m = udp_warp_matrix(center, size, rot, self.in_w, self.in_h)
        crop = cv2.warpAffine(img, m, (self.in_w, self.in_h), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))
        k_in = kpts[:, :2] @ m[:, :2].T + m[:, 2]
        if flip:
            crop = crop[:, ::-1]
            k_in[:, 0] = (self.in_w - 1) - k_in[:, 0]
            for l, r in PT18_FLIP_PAIRS:
                k_in[[l, r]] = k_in[[r, l]]
                vis[[l, r]] = vis[[r, l]]

        if self.train:
            crop = self._appearance(np.ascontiguousarray(crop), rng)
        target, weight = udp_heatmaps(k_in, vis, (self.in_w, self.in_h), (self.hm_w, self.hm_h), self.sigma)
        x = ((crop.astype(np.float32) - self.mean) / self.std).transpose(2, 0, 1)
        meta = {"center": center, "size": size, "index": i}
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(target), torch.from_numpy(weight), meta

    def _appearance(self, crop: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        a = self.aug
        crop = _photometric(crop, rng, a["photometric"])
        if rng.random() < a["grayscale"]:
            g = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
            crop = np.repeat(g[..., None], 3, axis=2)
        if rng.random() < a["lowres"]:
            f = rng.uniform(2.0, 4.0)
            small = cv2.resize(crop, (max(8, int(self.in_w / f)), max(8, int(self.in_h / f))), interpolation=cv2.INTER_AREA)
            crop = cv2.resize(small, (self.in_w, self.in_h), interpolation=cv2.INTER_LINEAR)
        if rng.random() < a["blur"]:
            crop = cv2.GaussianBlur(crop, (0, 0), rng.uniform(0.5, 2.0))
        if rng.random() < a["median_blur"]:
            crop = cv2.medianBlur(crop, int(rng.choice([3, 5])))
        if rng.random() < a["dropout"]:
            hh = int(rng.uniform(0.05, a["dropout_max"]) * self.in_h)
            ww = int(rng.uniform(0.05, a["dropout_max"]) * self.in_w)
            y0 = int(rng.integers(0, self.in_h - hh + 1))
            x0 = int(rng.integers(0, self.in_w - ww + 1))
            crop = crop.copy()
            crop[y0:y0 + hh, x0:x0 + ww] = 0
        return crop


def collate(batch):
    xs, ts, ws, metas = zip(*batch)
    return (torch.stack(xs), torch.stack(ts), torch.stack(ws),
            {"center": np.stack([m["center"] for m in metas]), "size": np.stack([m["size"] for m in metas]),
             "index": np.array([m["index"] for m in metas])})

In [ ]:
%%writefile /content/upar/train.py
"""Fine-tune ViTPose on UPAR-Pose with PoseTrack18 slots (Colab).

Recipe from MMPose's ViTPose COCO configs: AdamW, weight decay 0.1 (none on
bias/norm/pos_embed), layer-wise lr decay, drop path, grad clip 1.0, linear
warm-up; cosine decay as in the organizers' UPAR-Pose runs. bf16 autocast.

Resumable: the full training state is saved to the run directory (Google Drive)
every `ckpt_every` epochs, the best weights (by validation OKS-AP) whenever they
improve, and one JSON line per epoch is appended to log.jsonl.
"""
from __future__ import annotations

import json
import math
import os
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
from PIL import Image

import coco_kpt_eval as ce
import pose_infer as pi
import vitpose_model as vm
from data import PT18_FLIP_PAIRS, PoseDataset, collate

DEFAULTS = dict(
    run_name="vitb_random5",
    preset="base",
    init="hf:usyd-community/vitpose-base",  # or "mmpose:<url or path>"
    split="random5",                          # random5 | lodo_market | lodo_pa100k | lodo_peta | all
    epochs=30,
    batch_size=64,
    lr=5e-4,
    layer_decay=0.75,
    weight_decay=0.1,
    warmup_iters=500,
    min_lr_ratio=0.01,
    drop_path=0.3,
    clip_grad=1.0,
    eval_every=2,
    eval_flip=True,
    ckpt_every=5,
    workers=None,
    aug={},
    seed=0,
    input_size=(256, 192),     # (height, width); 384x288 starts from the 256x192 COCO checkpoint (pos_embed resized)
    heatmap_sigma=2.0,         # MMPose: 2 at 64x48 heatmaps, 3 at 96x72
    joint_weights=None,        # 17 per-slot loss weights, e.g. 1.5 on nose/head_bottom/head_top (slots 0-2)
    grad_checkpoint=False,     # recompute activations in backward: ~30% slower, much less memory
    max_iters_per_epoch=None,  # for smoke tests
    max_val=None,              # cap validation boxes (smoke tests)
    exclude_file="/content/upar/exclude_ann_ids.json",  # annotations removed from TRAINING only (decision 08/10)
    data_dir="/content/data",
    out_root="/content/drive/MyDrive/upar_track3/runs",
)

MMPOSE_URLS = {
    "base": "https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/td-hm_ViTPose-base_8xb64-210e_coco-256x192-216eae50_20230314.pth",
    "large": "https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/td-hm_ViTPose-large_8xb64-210e_coco-256x192-53609f55_20230314.pth",
    "huge": "https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/td-hm_ViTPose-huge_8xb64-210e_coco-256x192-e32adcd4_20230314.pth",
}
INPUT_CFG = {"input_height": 256, "input_width": 192, "image_mean": [0.485, 0.456, 0.406],
             "image_std": [0.229, 0.224, 0.225], "rescale_factor": 1 / 255.0, "normalize_factor": 200.0,
             "padding_factor": 1.25, "num_keypoints": 17}


def input_cfg(cfg: dict) -> dict:
    return {**INPUT_CFG, "input_height": int(cfg["input_size"][0]), "input_width": int(cfg["input_size"][1])}


def log(msg: str) -> None:
    print(time.strftime("%H:%M:%S"), msg, flush=True)


# ---------------------------------------------------------------- data

def load_split(cfg: dict):
    data_dir = cfg["data_dir"]
    ann = json.load(open(f"{data_dir}/annotations/train_upar_all_fixed.json"))
    if cfg["split"] == "all":
        train_ids, val_ids = {im["id"] for im in ann["images"]}, set()
    else:
        split = json.load(open(f"{data_dir}/annotations/splits.json"))[cfg["split"]]
        train_ids, val_ids = set(split["train"]), set(split["val"])
    needed = [im for im in ann["images"] if im["id"] in train_ids | val_ids]
    index = {im["id"]: i for i, im in enumerate(needed)}

    def read(im):
        return np.asarray(Image.open(f"{data_dir}/images/{im['file_name']}").convert("RGB"), dtype=np.uint8)

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=16) as ex:
        images = list(ex.map(read, needed))
    log(f"loaded {len(images)} images into RAM ({sum(i.nbytes for i in images) / 1e9:.2f} GB, {time.time() - t0:.0f}s)")

    def usable(a):
        return a["bbox"][2] > 1 and a["bbox"][3] > 1 and a.get("num_keypoints", 1) > 0

    def entries(ids):
        return [{"image_index": index[a["image_id"]], "bbox": a["bbox"], "keypoints": a["keypoints"], "ann": a}
                for a in ann["annotations"] if a["image_id"] in ids and usable(a)]

    excluded = set()
    if cfg.get("exclude_file"):
        if not os.path.isfile(cfg["exclude_file"]):
            raise FileNotFoundError(f"exclude_file {cfg['exclude_file']} not found (set exclude_file=None to train on all labels)")
        excluded = set(json.load(open(cfg["exclude_file"]))["ann_ids"])
    train_entries = [e for e in entries(train_ids) if e["ann"]["id"] not in excluded]
    log(f"excluded {len(entries(train_ids)) - len(train_entries)} suspicious annotations from training "
        f"(list has {len(excluded)}; validation keeps all labels)")
    val_gt = {"images": [im for im in needed if im["id"] in val_ids],
              "annotations": [a for a in ann["annotations"] if a["image_id"] in val_ids],
              "categories": ann["categories"]}
    return images, train_entries, entries(val_ids), val_gt


# ---------------------------------------------------------------- model / optimisation

def build_model(cfg: dict) -> torch.nn.Module:
    model = vm.build(cfg["preset"], drop_path_rate=cfg["drop_path"], img_size=tuple(int(v) for v in cfg["input_size"]))
    model.grad_checkpoint = bool(cfg["grad_checkpoint"])
    kind, _, ref = cfg["init"].partition(":")
    if kind == "hf":
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        state = vm.from_hf(load_file(hf_hub_download(ref, "model.safetensors")), model.depth)
    elif kind == "mmpose":
        url = ref or MMPOSE_URLS[cfg["preset"]]
        path = url
        if url.startswith("http"):
            path = f"/content/ckpt/{os.path.basename(url)}"
            if not os.path.isfile(path):
                os.makedirs(os.path.dirname(path), exist_ok=True)
                log(f"downloading {url}")
                urllib.request.urlretrieve(url, path + ".part")
                os.replace(path + ".part", path)
        state = vm.from_mmpose(vm.load_checkpoint_file(path), model.depth)
    elif kind in ("pt", "drive", "file"):
        path = ref
        if kind == "drive" and not path.startswith("/"):
            path = f"/content/drive/MyDrive/upar_track3/runs/{ref}"
        state = torch.load(path, map_location="cpu", weights_only=False)
        if "model" in state:
            state = state["model"]
        state = {k: v.float() if v.is_floating_point() else v for k, v in state.items()}
        if "pos_embed" in state:
            state["pos_embed"] = vm.resize_pos_embed(state["pos_embed"], model.grid)
        model.load_state_dict(state, strict=True)
        return model
    else:
        raise ValueError(f"unknown init {cfg['init']}")
    state["pos_embed"] = vm.resize_pos_embed(state["pos_embed"], model.grid)
    model.load_state_dict(vm.coco_head_to_pt18(state), strict=True)
    return model


def param_groups(model, cfg: dict):
    groups = {}
    num_layers = model.depth + 1
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        no_decay = p.ndim == 1 or name == "pos_embed"
        lid = model.layer_id(name)
        key = (lid, no_decay)
        if key not in groups:
            groups[key] = {"params": [], "weight_decay": 0.0 if no_decay else cfg["weight_decay"],
                           "lr_scale": cfg["layer_decay"] ** (num_layers - lid)}
        groups[key]["params"].append(p)
    return list(groups.values())


def lr_factor(it: int, total: int, cfg: dict) -> float:
    if it < cfg["warmup_iters"]:
        return 0.001 + (1 - 0.001) * it / cfg["warmup_iters"]
    t = (it - cfg["warmup_iters"]) / max(1, total - cfg["warmup_iters"])
    return cfg["min_lr_ratio"] + (1 - cfg["min_lr_ratio"]) * 0.5 * (1 + math.cos(math.pi * min(t, 1.0)))


# ---------------------------------------------------------------- evaluation

@torch.no_grad()
def evaluate(model, loader, entries, val_gt, flip: bool, device) -> dict:
    model.eval()
    preds = []
    for x, _, _, meta in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"):
            hm = model(x)
            if flip:
                hm_f = model(torch.flip(x, dims=[3]))
        hm = hm.float().cpu().numpy()
        if flip:
            hm = 0.5 * (hm + pi.flip_back(hm_f.float().cpu().numpy(), PT18_FLIP_PAIRS, shift=False))
        coords, maxvals = pi.decode_heatmaps(hm, meta["center"], meta["size"] / 200.0)
        for j, idx in enumerate(meta["index"]):
            a = entries[idx]["ann"]
            k = np.concatenate([coords[j], maxvals[j][:, None]], axis=1).reshape(-1).tolist()
            preds.append({"image_id": a["image_id"], "category_id": 1, "box_id": a["id"], "keypoints": k,
                          "score": pi.person_score(maxvals[j], "mean_all")})
    model.train()
    metrics = ce.evaluate(val_gt, preds)
    per_joint = ce.per_keypoint_similarity(val_gt, preds)
    metrics["pck"] = {k: round(v["pck50"], 4) for k, v in per_joint.items()}
    metrics["head_oks"] = float(np.mean([per_joint[k]["oks"] for k in ("nose", "head_bottom", "head_top") if k in per_joint]))
    metrics["lr_swap_rate"] = lr_swap_rate(val_gt, preds)
    return metrics


def lr_swap_rate(gt: dict, preds: list[dict], min_joints: int = 4) -> float:
    """Share of people whose limb joints (slots 5-16) agree better with the ground truth after
    swapping left/right: a front/back confusion that costs the whole pose (failure mode #3)."""
    var = (ce.DEFAULT_SIGMAS * 2) ** 2
    anns = {a["id"]: a for a in gt["annotations"]}
    swap = np.arange(17)
    for l, r in PT18_FLIP_PAIRS:
        swap[l], swap[r] = r, l
    limbs = np.arange(5, 17)
    n = swapped = 0
    for d in preds:
        a = anns.get(d["box_id"])
        if a is None:
            continue
        g = np.asarray(a["keypoints"], dtype=np.float64).reshape(17, 3)
        k = np.asarray(d["keypoints"], dtype=np.float64).reshape(17, 3)[:, :2]
        vis = limbs[(g[limbs, 2] > 0) & (g[swap[limbs], 2] > 0)]
        if len(vis) < min_joints:
            continue
        area = a["area"] + np.spacing(1)
        same = np.exp(-((k[vis] - g[vis, :2]) ** 2).sum(1) / var[vis] / area / 2).mean()
        flip = np.exp(-((k[vis] - g[swap[vis], :2]) ** 2).sum(1) / var[vis] / area / 2).mean()
        n += 1
        swapped += flip > same
    return round(swapped / max(n, 1), 4)


# ---------------------------------------------------------------- main loop

def train(user_cfg: dict) -> dict:
    cfg = {**DEFAULTS, **user_cfg}
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_dir = f"{cfg['out_root']}/{cfg['run_name']}"
    os.makedirs(run_dir, exist_ok=True)
    json.dump(cfg, open(f"{run_dir}/config.json", "w"), indent=1)
    log(f"run {cfg['run_name']} -> {run_dir}")

    images, train_entries, val_entries, val_gt = load_split(cfg)
    if cfg["max_val"]:
        val_entries = val_entries[:cfg["max_val"]]
        keep = {e["ann"]["image_id"] for e in val_entries}
        val_gt = {**val_gt, "images": [im for im in val_gt["images"] if im["id"] in keep],
                  "annotations": [a for a in val_gt["annotations"] if a["image_id"] in keep]}
    log(f"train {len(train_entries)} boxes | val {len(val_entries)} boxes")
    workers = cfg["workers"] if cfg["workers"] is not None else min(24, os.cpu_count() or 2)
    in_cfg = input_cfg(cfg)
    train_ds = PoseDataset(images, train_entries, in_cfg, train=True, aug=cfg["aug"], sigma=cfg["heatmap_sigma"])
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True,
                                           num_workers=workers, collate_fn=collate, pin_memory=True,
                                           persistent_workers=workers > 0)
    val_dl = None
    if val_entries:
        val_ds = PoseDataset(images, val_entries, in_cfg, train=False, sigma=cfg["heatmap_sigma"])
        val_dl = torch.utils.data.DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=workers,
                                             collate_fn=collate, pin_memory=True)

    model = build_model(cfg).to(device)
    joint_w = None
    if cfg["joint_weights"] is not None:
        joint_w = torch.tensor(cfg["joint_weights"], dtype=torch.float32, device=device)
        if joint_w.shape != (17,):
            raise ValueError("joint_weights needs 17 values (PoseTrack18 slots)")
    opt = torch.optim.AdamW(param_groups(model, cfg), lr=cfg["lr"], betas=(0.9, 0.999))
    iters_per_epoch = len(train_dl) if not cfg["max_iters_per_epoch"] else min(len(train_dl), cfg["max_iters_per_epoch"])
    total_iters = iters_per_epoch * cfg["epochs"]

    start_epoch, it, best_ap = 0, 0, -1.0
    last_path = f"{run_dir}/last.pt"
    if os.path.isfile(last_path):
        state = torch.load(last_path, map_location="cpu", weights_only=False)
        for key in ("preset", "input_size", "split"):
            old, new = state["cfg"].get(key, DEFAULTS[key]), cfg[key]
            if json.dumps(old) != json.dumps(list(new) if isinstance(new, tuple) else new):
                raise ValueError(f"{last_path} was trained with {key}={old}, config says {new}: use a new run_name")
        model.load_state_dict(state["model"])
        opt.load_state_dict(state["opt"])
        start_epoch, it, best_ap = state["epoch"] + 1, state["iter"], state["best_ap"]
        log(f"resumed from epoch {state['epoch']} (best AP {best_ap:.4f})")

    model.train()
    for epoch in range(start_epoch, cfg["epochs"]):
        t0, loss_sum, n = time.time(), 0.0, 0
        for step, (x, target, weight, _) in enumerate(train_dl):
            if step >= iters_per_epoch:
                break
            f = lr_factor(it, total_iters, cfg)
            for g in opt.param_groups:
                g["lr"] = cfg["lr"] * f * g["lr_scale"]
            x, target, weight = (t.to(device, non_blocking=True) for t in (x, target, weight))
            if joint_w is not None:
                weight = weight * joint_w
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"):
                out = model(x)
            loss = (((out.float() - target) ** 2) * weight[:, :, None, None]).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip_grad"])
            opt.step()
            loss_sum += loss.item()
            n += 1
            it += 1
        dt = time.time() - t0
        record = {"epoch": epoch, "iter": it, "loss": loss_sum / max(n, 1), "lr": cfg["lr"] * lr_factor(it, total_iters, cfg),
                  "sec": round(dt, 1), "img_per_s": round(n * cfg["batch_size"] / dt, 1)}

        last_epoch = epoch == cfg["epochs"] - 1
        if val_dl is not None and ((epoch + 1) % cfg["eval_every"] == 0 or last_epoch):
            metrics = evaluate(model, val_dl, val_entries, val_gt, cfg["eval_flip"], device)
            record.update({k: round(metrics[k], 4) for k in ("AP", "AP50", "AP75", "AP_medium", "AP_large", "AR_large")})
            record["lr_swap"] = metrics["lr_swap_rate"]
            record["head_oks"] = round(metrics["head_oks"], 4)
            record["pck"] = {k: metrics["pck"].get(k) for k in ("nose", "head_top", "left_wrist", "right_wrist")}
            if metrics["AP"] > best_ap:
                best_ap = metrics["AP"]
                torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()},
                           f"{run_dir}/best_fp16.pt")
                record["best"] = True
        elif val_dl is None and last_epoch:
            torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()},
                       f"{run_dir}/best_fp16.pt")
        with open(f"{run_dir}/log.jsonl", "a") as fh:
            fh.write(json.dumps(record) + "\n")
        log(json.dumps(record))

        if (epoch + 1) % cfg["ckpt_every"] == 0 or last_epoch:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "iter": it,
                        "best_ap": best_ap, "cfg": cfg}, last_path + ".tmp")
            os.replace(last_path + ".tmp", last_path)

    export = {"preset": cfg["preset"], "init": cfg["init"], "split": cfg["split"], "slots": "posetrack18",
              "flip_pairs": [list(p) for p in PT18_FLIP_PAIRS], "flip_shift": False, "best_val_ap": best_ap,
              **in_cfg}
    json.dump(export, open(f"{run_dir}/model_config.json", "w"), indent=1)
    log(f"done. best val AP {best_ap:.4f}. weights: {run_dir}/best_fp16.pt")
    return {"run_dir": run_dir, "best_ap": best_ap}

In [ ]:
%%writefile /content/upar/exclude_ann_ids.json
{
 "description": "Annotations excluded from training (decision 08/10/2026): L/R order of shoulders vs hips disagrees (likely half-body L/R swap) or head_top below head_bottom. Source: prepared/suspicious_labels.csv. limb_asym cases are kept (mostly real foreshortening).",
 "reasons": [
  "lr_shoulder_hip",
  "head_order"
 ],
 "counts": {
  "head_order/market": 1,
  "head_order/pa100k": 4,
  "lr_shoulder_hip/market": 96,
  "lr_shoulder_hip/pa100k": 41,
  "lr_shoulder_hip/peta": 34
 },
 "ann_ids": [
  412,
  459,
  1068,
  1120,
  1276,
  1397,
  1788,
  2048,
  2152,
  4029,
  4705,
  5004,
  5141,
  5797,
  6252,
  7501,
  7506,
  7570,
  8111,
  8242,
  8305,
  8736,
  8848,
  8909,
  10492,
  10814,
  12134,
  13073,
  13246,
  13247,
  13665,
  13666,
  13769,
  17980,
  19644,
  20442,
  20864,
  21261,
  22361,
  22817,
  22935,
  22967,
  23050,
  23095,
  23247,
  25124,
  25280,
  25284,
  25398,
  25437,
  25486,
  25503,
  25535,
  25589,
  26151,
  28535,
  28536,
  28647,
  28649,
  29105,
  31183,
  31212,
  31512,
  31748,
  31783,
  31798,
  31939,
  32020,
  32168,
  32367,
  33395,
  33418,
  34012,
  34538,
  37732,
  37754,
  38075,
  40754,
  40821,
  66337,
  66361,
  66430,
  66465,
  66654,
  66667,
  66668,
  66717,
  66997,
  67063,
  67442,
  67524,
  67525,
  68928,
  70181,
  70287,
  70315,
  70365,
  70422,
  70498,
  70499,
  70523,
  70529,
  70798,
  70800,
  70801,
  70802,
  70803,
  70812,
  70844,
  70872,
  70973,
  70977,
  71054,
  71091,
  71094,
  71110,
  71120,
  71143,
  71144,
  71666,
  72276,
  72466,
  72547,
  72695,
  72753,
  73244,
  73316,
  73326,
  73447,
  73452,
  73462,
  73478,
  73542,
  73584,
  73625,
  73752,
  73786,
  73809,
  73884,
  73964,
  74034,
  74061,
  74133,
  75168,
  75202,
  75247,
  75382,
  75391,
  75414,
  75480,
  75504,
  75574,
  75701,
  75730,
  75732,
  75736,
  75737,
  75778,
  75818,
  75857,
  75860,
  75886,
  75888,
  76001,
  76015,
  76296,
  76525,
  76600,
  76610,
  76686,
  77020,
  77089,
  77261,
  78038,
  78041,
  78161
 ]
}

In [ ]:
# 2. CẤU HÌNH — ViTPose-Huge 384x288 (637M params) + Loss x1.8 vùng đầu
#   Kết hợp sức chứa mô hình tối đa (Huge: 637M params) với độ phân giải cao nhất (384x288, token grid 24x18).
#   Khởi tạo:
#     - Mặc định: "mmpose:" tải checkpoint COCO chính thức của ViTPose-Huge (pos_embed tự động resize lên 24x18).
#     - Hoặc fine-tune tiếp từ weights Huge 256 đã có trên Drive: init="drive:r06_vith_random5/best_fp16.pt"
#   Tăng trọng số loss 3 khớp đầu (1.8) để tối ưu OKS head.
#   batch_size=32 + grad_checkpoint=True chạy an toàn trên mọi GPU Colab (A100 40GB/80GB, L4 24GB, G4 102GB).
#   Ước lượng 20 epoch: G4 ~90–110 phút | A100 ~120–140 phút.
FOLDS = ["random5"]
CONFIG = dict(
    run_name="r09_vith384",  # -> runs/r09_vith384_random5
    preset="huge",
    init="mmpose:",          # hoặc "drive:r06_vith_random5/best_fp16.pt" nếu muốn học tiếp từ Huge 256
    input_size=(384, 288),
    heatmap_sigma=3.0,
    joint_weights=[1.8, 1.8, 1.8] + [1.0] * 14,
    grad_checkpoint=True,    # bật để chạy an toàn trên GPU 24GB–40GB
    epochs=20,
    batch_size=32,
    lr=1.5e-4,
    layer_decay=0.85,        # MMPose ViTPose-H
    drop_path=0.55,          # MMPose ViTPose-H
    eval_every=2,
)

In [ ]:
# 3. Train (tự resume nếu run_name đã có last.pt trên Drive). Log: Drive/upar_track3/runs/<run_name>/log.jsonl
sys.path[:0] = [WORK]
import importlib, vitpose_model, pose_infer, coco_kpt_eval, data, train as T
for m in (vitpose_model, pose_infer, coco_kpt_eval, data, T):
    importlib.reload(m)
results = {}
for fold in FOLDS:
    r = T.train({**CONFIG, "split": fold, "run_name": f"{CONFIG['run_name']}_{fold}"})
    results[fold] = r
    print("RESULT", fold, json.dumps(r))
lodo = [results[f]["best_ap"] for f in FOLDS if f.startswith("lodo_")]
summary = {"config": CONFIG, "folds": {f: results[f]["best_ap"] for f in FOLDS}}
if lodo:
    summary["lodo_mean_ap"] = float(sum(lodo) / len(lodo))
    print(f"\n==> LODO mean AP = {summary['lodo_mean_ap']:.4f} | " + " ".join(f"{f}={results[f]['best_ap']:.4f}" for f in FOLDS))
json.dump(summary, open(f"{CONFIG.get('out_root', T.DEFAULTS['out_root'])}/{CONFIG['run_name']}_summary.json", "w"), indent=1)
result = results[FOLDS[-1]]

In [ ]:
# 4. Đo trên bộ test giả lập. FOLDS = ["random5"]: mốc so sánh (flip) r03 L256 = 0.9588, r05 L384 = 0.9590.
#    FOLDS = ["all"]: mock nằm TRONG dữ liệu train -> chỉ là kiểm tra không hỏng (điểm phải >= bản random5). LODO: bỏ qua.
if FOLDS not in (["random5"], ["all"]):
    raise SystemExit("mock eval skipped (only for FOLDS=['random5'] or ['all']) -> nhớ Ctrl+S")
if FOLDS == ["all"]:
    print("CHÚ Ý: model train trên cả ảnh của mock test -> điểm dưới đây lạc quan, chỉ để kiểm tra model không hỏng")
import numpy as np
from PIL import Image
import vitpose_model as vm, pose_infer as pi, coco_kpt_eval as ce

run_dir = result["run_dir"]
mc = json.load(open(f"{run_dir}/model_config.json"))
model = vm.build_for_config(mc).eval().cuda()
model.load_state_dict({k: v.float() for k, v in torch.load(f"{run_dir}/best_fp16.pt", map_location="cpu").items()})

@torch.no_grad()
def forward(batch):
    return model(torch.from_numpy(batch).cuda()).float().cpu().numpy()

mock_root = "/content/mock_test/random5"
if not os.path.isdir(mock_root):
    subprocess.run([sys.executable, f"{WORK}/build_mock_test.py", "--data-dir", f"{DATA}/images",
                    "--ann", f"{DATA}/annotations/train_upar_all_fixed.json", "--splits", f"{DATA}/annotations/splits.json",
                    "--split", "random5", "--out-dir", "/content/mock_test"], check=True)
subsets = sorted(d for d in os.listdir(mock_root) if os.path.isdir(f"{mock_root}/{d}/input_data"))
summary = {}
for flip in (False, True):
    aps = {}
    for sub in subsets:
        boxes = json.load(open(f"{mock_root}/{sub}/input_data/boxes.json"))
        gt = json.load(open(f"{mock_root}/{sub}/ground_truth.json"))
        by_img = {}
        for a in boxes["annotations"]:
            by_img.setdefault(a["image_id"], []).append(a)
        preds = []
        for im in boxes["images"]:
            anns = by_img.get(im["id"], [])
            if not anns:
                continue
            img = np.asarray(Image.open(f"{mock_root}/{sub}/input_data/images/{im['file_name']}").convert("RGB"), dtype=np.float32)
            k, s, ok = pi.predict_boxes(img, [a["bbox"] for a in anns], forward, mc, flip=flip, flip_pairs=pi.PT18_FLIP_PAIRS)
            for i, a in enumerate(anns):
                flat = np.concatenate([k[i], s[i][:, None]], 1).reshape(-1).tolist()
                preds.append({"image_id": im["id"], "category_id": 1, "box_id": a["id"], "keypoints": flat,
                              "score": pi.person_score(s[i], "mean_all")})
        aps[sub] = ce.evaluate(gt, preds)
        pj = ce.per_keypoint_similarity(gt, preds)
        aps[sub]["head_oks"] = float(np.mean([pj[k]["oks"] for k in ("nose", "head_bottom", "head_top")]))
    mean = float(np.mean([aps[s]["AP"] for s in subsets]))
    by_size = {m: float(np.mean([aps[s][m] for s in subsets if aps[s][m] > -1])) for m in ("AP_medium", "AP_large", "AR_large", "head_oks")}
    summary[f"flip={flip}"] = {"meanAP": mean, **by_size, **{s: aps[s]["AP"] for s in subsets}}
    print(f"flip={flip!s:5s} meanAP={mean:.4f} AP_medium={by_size['AP_medium']:.4f} AP_large={by_size['AP_large']:.4f} "
          f"headOKS={by_size['head_oks']:.4f} | "
          + " ".join(f"{s}={aps[s]['AP']:.4f}" for s in subsets))
json.dump(summary, open(f"{run_dir}/mock_eval.json", "w"), indent=1)
print(f"\nweights cho bài nộp: {run_dir}/best_fp16.pt + model_config.json  -> nhớ Ctrl+S")